# 01 — Demo Runbook (run 00_setup.ipynb first)

Dry-runs every demo beat from the backend so you know the stage is safe,
then points you at the Streamlit UI for the real performance.

**The three differentiators to show the judges:**
1. 🔄 Self-correction loop — Vikram's case resolves itself (cell 3)
2. ⚡ Live ROCm GPU telemetry — Streamlit Tab 3 while inference runs
3. 📦 Bulk import stress test — 20 customers live (cell 4, then Tab 2)

In [ ]:
# 1) Compile the LangGraph pipeline
from datetime import datetime
from graph import build_graph, complete_case
from state import create_initial_state

app = build_graph()
print("✅ 9-agent graph compiled — extract → id_verify → compliance ⇄ refine →")
print("   fan_out → [entity ∥ financial] → risk → (auto | human review)")

In [ ]:
# 2) Canonical clean customer — Priya Sharma → AUTO APPROVE in one pass.
#    Note the audit line where the DOB gate clears the exact-name decoy
#    ('Priya Sharma' IS on the RBI defaulters list — born 1978, not 1992).
state = create_initial_state(
    customer_id="DEMO-PRIYA", name="Priya Sharma", dob="1992-09-08",
    nationality="Indian", address="B-204 Green Park Bengaluru", pin_code="560034",
    occupation="software engineer", income=1_200_000,
    source_of_funds="Monthly salary from Infosys Limited",
    account_purpose="Savings and investments",
    aadhaar_path="./mock_data/images/priya_aadhaar.jpg",
    pan_path="./mock_data/images/priya_pan.jpg",
    received_at=datetime.now().strftime("%Y-%m-%dT%H:%M:%S IST"),
)
result = app.invoke(state)
print(f"Decision: {result['decision']}  |  score {result['risk_score']}  |  {result['routing']}\n")
print(*result["audit_log"], sep="\n")

In [ ]:
# 3) THE HEADLINE — Vikram Malhotra and the self-correction loop.
#    Demo line: \"The match starts at 0.81. The system doesn't halt — it
#    issues a Refinement Request, finds the father's name in the Aadhaar QR,
#    compares it to the listed individual's father, and resolves the case
#    autonomously.\" Then the 3-hop Orion chain still routes him to a human.
state = create_initial_state(
    customer_id="DEMO-VIKRAM", name="Vikram Malhotra", dob="1968-11-14",
    nationality="Indian", address="34 Golf Links New Delhi", pin_code="110003",
    occupation="business owner", income=12_000_000,
    source_of_funds="Business profits from multiple ventures",
    account_purpose="Business transactions and overseas investment",
    aadhaar_path="./mock_data/images/vikram_aadhaar.jpg",
    pan_path="./mock_data/images/vikram_pan.jpg",
    received_at=datetime.now().strftime("%Y-%m-%dT%H:%M:%S IST"),
)
# Father's name as the Aadhaar QR 'care_of' would carry it — the extraction
# agent only surfaces this on the refinement pass, so the loop has work to do.
state["declared"]["father_name"] = "Ramesh Malhotra"

result = app.invoke(state)
print(f"Decision: {result['decision']}  |  score {result['risk_score']}  |  "
      f"{result['routing']}  |  refinement passes: {result.get('refine_count', 0)}\n")
print(*result["audit_log"], sep="\n")

# Officer closes the case (in the UI this is the HITL panel)
closed = complete_case(result, "OFFICER-KYC-014", "HOLD_FOR_DOCUMENTS",
                       "3-hop corporate chain noted; requesting ITR + GST certificate.")
print(f"\nHITL closed → {closed['final_decision']} (source: {closed['decision_source']})")

In [ ]:
# 4) Bulk regression — all 20 customers must land 12 APPROVE / 5 REVIEW / 3 REJECT.
#    Run this before going on stage. 20/20 PASS = the demo climax is safe.
!python bulk_acceptance_test.py

## 5) Showtime — the Streamlit UI

```bash
streamlit run app.py
```

Suggested demo order:
1. **Tab 1** — Priya Sharma (type her declared data) → AUTO APPROVE ~30s.
2. **Tab 1** — Vikram Malhotra (fill the *Father's Name* field with
   `Ramesh Malhotra` if not uploading a real Aadhaar image) → watch the
   🔄 self-correction banner, the 3-hop entity chain, then close the case
   as the human officer.
3. **Tab 3** — leave telemetry visible while you start…
4. **Tab 2** — 🚀 Import All Customers: 20 cases stream through live on
   the MI300X → 12 ✅ / 5 ⚠️ / 3 ⛔.
